In [2]:
!pip install -q -U google-genai

In [6]:
from google import genai
from google.colab import userdata
from google.genai import types
import json
from pydantic import BaseModel
from typing import Optional

client=genai.Client(api_key=userdata.get("GEMINI_API_KEYS"))
MODEL="gemini-3.5-flash-lite"

In [8]:
user_input=input("Enter meeting info: ")

prompt=f"""
refer to user input meeting info:{user_input} ,extract name,meeting date,meeting time,meeting purpose.
return json format with keys name,date,time,purpose and valid, if all feilds(keys name,date,time,purpose) are mentioned properly then set valid key as true else set valid key as false.
date must in format date month(in words) year
Return JSON format
"""

class meeting(BaseModel):
  name:Optional[str]
  date:Optional[str]
  time:Optional[str]
  purpose:Optional[str]
  valid:bool

response=client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.8,
        max_output_tokens=512,
        response_mime_type="application/json",
        response_schema=meeting,
        thinking_config=types.ThinkingConfig(thinking_level='low')
    )
)
data=json.loads(response.text)
keys=['name','date','time','purpose','valid']
for key in keys:
  data.setdefault(key,None)

print(json.dumps(data,indent=2))

Enter meeting info: Book a meeting with Rahul  on 31/10/2026 at 3 PM for genai classes discussion.
{
  "name": "Rahul",
  "date": "31 October 2026",
  "time": "3 PM",
  "purpose": "genai classes discussion",
  "valid": true
}
